# 네이버 영화리뷰 감성 분석

### 01. 데이터 로드

**✅ 실습 과제**
- [네이버 영화 리뷰 데이터셋](https://raw.githubusercontent.com/e9t/nsmc/master/ratings.txt)을 불러온다.
- 데이터의 컬럼 구성과 샘플을 확인한다.

**🔍 확인 질문**
1. 리뷰 텍스트와 정답 라벨은 각각 어떤 컬럼에 저장되어 있는가?  -> 리뷰 텍스트 : document, 정답라벨: label
2. 긍정 / 부정 라벨은 어떤 값으로 표현되어 있는가?  -> 긍정:1, 부정:0

**Git Link**  
https://github.com/e9t/nsmc

In [2]:
# 데이터 다운로드
from tensorflow.keras.utils import get_file

ratings_train_path = get_file("ratings_train.txt", "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt")
ratings_test_path = get_file("ratings_test.txt", "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt")

ratings_train_path, ratings_test_path

2026-03-05 20:11:46.909645: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


('/Users/kimunoo/.keras/datasets/ratings_train.txt',
 '/Users/kimunoo/.keras/datasets/ratings_test.txt')

In [3]:
import pandas as pd

# 데이터프레임 생성
ratings_df = pd.read_csv('data/ratings.txt', sep='\t')

In [4]:
# 데이터 샘플링
ratings_df.head()

,id,document,label
0,8112052,어릴때보고 지금다시봐도 재밌어요ㅋㅋ,1
1,8132799,"디자인을 배우는 학생으로, 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산...",1
2,4655635,폴리스스토리 시리즈는 1부터 뉴까지 버릴께 하나도 없음.. 최고.,1
3,9251303,와.. 연기가 진짜 개쩔구나.. 지루할거라고 생각했는데 몰입해서 봤다.. 그래 이런...,1
4,10067386,안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화.,1


### 02. 데이터 전처리

##### 02-01. 한글 전처리

**✅ 실습 과제**
- 특수문자, 숫자, 불필요한 기호를 제거한다.
- 정규표현식을 사용하여 한글만 남긴다.

**🔍 확인 질문**
1. 한글 전처리를 하지 않으면 어떤 문제가 발생할 수 있는가?
2. 감성 분석에서 이모지나 느낌표는 제거하는 것이 항상 옳은가?




In [5]:
# 결측치 제거
ratings_df.isna().sum()

ratings_df = ratings_df.dropna(subset=['document'])
ratings_df.isna().sum()

id          0
document    0
label       0
dtype: int64

In [ ]:
import re

def clean_text(text):
    # 1. 문자가 아닌 경우(NaN 등) 빈 문자열로 처리
    if not isinstance(text, str):
        return ""
    
    # 2. 한글과 공백만 남기고 제거 (특수문자, 숫자 등 삭제)
    text = re.sub(r'[^가-힣\s]', '', text)
    
    # 3. 양끝 공백 제거
    text = text.strip()
    
    return text


# 'document' 컬럼의 각 데이터를 clean_text 함수에 집어넣어라!
ratings_df['document'] = ratings_df['document'].apply(clean_text)

# 결과 확인 (빈 문자열이 된 행이 있을 수 있으니 다시 한번 체크)
print(ratings_df.head())

In [6]:
import re
from nltk.tokenize import sent_tokenize, word_tokenize
from konlpy.tag import Okt

# 한글 토큰화 전처리 (특수문자 처리, 어간 추출, 불용어 처리) -> 함수
stop_words = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']

# corpus 만들기
corpus = " ".join(ratings_df['document'].tolist())
print(len(corpus))
print(corpus[:500])

corpus = re.sub(r'[^가-힣 ]','',corpus)

print(corpus[:10000])

7242878
어릴때보고 지금다시봐도 재밌어요ㅋㅋ 디자인을 배우는 학생으로, 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산업이 부러웠는데. 사실 우리나라에서도 그 어려운시절에 끝까지 열정을 지킨 노라노 같은 전통이있어 저와 같은 사람들이 꿈을 꾸고 이뤄나갈 수 있다는 것에 감사합니다. 폴리스스토리 시리즈는 1부터 뉴까지 버릴께 하나도 없음.. 최고. 와.. 연기가 진짜 개쩔구나.. 지루할거라고 생각했는데 몰입해서 봤다.. 그래 이런게 진짜 영화지 안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화. 사랑을 해본사람이라면 처음부터 끝까지 웃을수 있는영화 완전 감동입니다 다시봐도 감동 개들의 전쟁2 나오나요? 나오면 1빠로 보고 싶음 굿 바보가 아니라 병 쉰 인듯 내 나이와 같은 영화를 지금 본 나는 감동적이다..하지만 훗날 다시보면대사하나하나 그 감정을완벽하게 이해할것만 같다... 재밌다 고질라니무 귀엽다능ㅋㅋ 영화의 오페라화라고 해야할 작품. 극단적 평갈림은 어쩔 수 없는 듯. 3도 반전 
어릴때보고 지금다시봐도 재밌어요 디자인을 배우는 학생으로 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산업이 부러웠는데 사실 우리나라에서도 그 어려운시절에 끝까지 열정을 지킨 노라노 같은 전통이있어 저와 같은 사람들이 꿈을 꾸고 이뤄나갈 수 있다는 것에 감사합니다 폴리스스토리 시리즈는 부터 뉴까지 버릴께 하나도 없음 최고 와 연기가 진짜 개쩔구나 지루할거라고 생각했는데 몰입해서 봤다 그래 이런게 진짜 영화지 안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화 사랑을 해본사람이라면 처음부터 끝까지 웃을수 있는영화 완전 감동입니다 다시봐도 감동 개들의 전쟁 나오나요 나오면 빠로 보고 싶음 굿 바보가 아니라 병 쉰 인듯 내 나이와 같은 영화를 지금 본 나는 감동적이다하지만 훗날 다시보면대사하나하나 그 감정을완벽하게 이해할것만 같다 재밌다 고질라니무 귀엽다능 영화의 오페라화라고 해야할 작품 극단적 평갈림은 어쩔 수 없는 듯 도 반전 좋았제  평점 왜 낮아 긴장감 스릴감

##### 02-02. Tokenizing & Sequencing

**✅ 실습 과제**
- Tokenizer를 사용해 단어 사전을 생성한다.
- 문장을 정수 시퀀스로 변환한다.
- padding을 적용하여 시퀀스 길이를 맞춘다.

**🔍 확인 질문**
1. `num_words` 파라미터는 어떤 역할을 하는가?
2. padding을 하지 않으면 배치 학습에서 어떤 문제가 발생하는가?



In [7]:
# sequence 작업 (단어사전 생성, 텍스트-수열 변환)
from torchtext.vocab import build_vocab_from_iterator

# 값을 하나씩 반환하는 함수
def yield_tokens(data):
    for tokens in data:
        yield tokens

ModuleNotFoundError: No module named 'torchtext'

In [ ]:
# padding 작업

##### 02-03. Sequence Decoding

**✅ 실습 과제**
- 정수 시퀀스를 다시 텍스트로 복원해본다.
- 토큰 인덱스와 단어의 매핑 관계를 확인한다.

**🔍 확인 질문**
1. `<OOV>` 토큰은 언제 사용되는가?
2. 디코딩 결과가 원문과 완전히 동일하지 않은 이유는 무엇인가?


### 03. 모델 생성 및 학습

**✅ 실습 과제**
- Embedding Layer를 포함한 감성분석 모델을 정의한다.
- 손실 함수와 옵티마이저를 설정한다.
- 학습 과정을 통해 loss 변화를 확인한다.

**🔍 확인 질문**
1. Embedding Layer는 어떤 역할을 하는가?
2. One-hot 인코딩 대신 Embedding을 사용하는 이유는 무엇인가?

In [ ]:
# 모델 정의

In [ ]:
# 모델 인스턴스 생성

In [ ]:
# 학습

### 04. 모델 평가

**✅ 실습 과제**
- 검증 데이터로 모델 성능을 평가한다.
- 정확도(acc)와 손실(loss)을 확인한다.

**🔍 확인 질문**
1. 학습 데이터 성능과 평가 데이터 성능 차이가 의미하는 것은 무엇인가?
2. 과적합 여부는 어떻게 판단할 수 있는가?

### 05. 모델 추론

**✅ 실습 과제**
- 임의의 문장을 입력하여 감성을 예측한다.
- 출력 확률을 기반으로 긍/부정을 해석한다.

**🔍 확인 질문**
1. 모델 출력값은 확률인가 점수인가?
2. 기준값(threshold)은 왜 필요한가?